In [5]:
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [6]:
class SpatioTemporalGNN(nn.Module):
    def __init__(self,nodes,input_features,lookback,hidden_dims):
        super(SpatioTemporalGNN,self).__init__()
        self.nodes=nodes
        self.lookback=lookback
        self.hidden_dims=hidden_dims

        self.gcn_weight=nn.Parameter(torch.FloatTensor(input_features,hidden_dims))
        nn.init.xavier_uniform_(self.gcn_weight)

        self.lstm=nn.LSTM(input_size=hidden_dims,hidden_size=hidden_dims,num_layers=1,batch_first=True)
        self.fc=nn.Linear(hidden_dims*lookback,1)

    def forward(self, x, adj):
        # Input 'x' structural footprint: [Batch, Features, Nodes, Timesteps]
        batch_size, _, num_nodes, timesteps = x.shape
        A = torch.tensor(adj, dtype=torch.float32, device=x.device)
        
        # 1. Spatial Processing Feature Loop
        spatial_features = []
        for t in range(timesteps):
            xt = x[:, :, :, t].permute(0, 2, 1)  # Shape becomes: [Batch, Nodes, Features]
            spatial_kernel = torch.matmul(A, xt)  # Aggregate spatial neighbors
            z = torch.matmul(spatial_kernel, self.gcn_weight)  # Map features via learnable weights [B, N, H]
            spatial_features.append(z.unsqueeze(3))
            
        spatial_out = torch.cat(spatial_features, dim=3)  # Recompiled output dimensions: [B, N, H, T]
    
        # 2. Temporal Sequencer Loop
        outputs = []
        for n in range(num_nodes):
            node_seq = spatial_out[:, n, :, :].permute(0, 2, 1)  # Extract per-node sequences [Batch, Timesteps, Hidden]
            lstm_out, _ = self.lstm(node_seq)
            lstm_flat = lstm_out.reshape(batch_size, -1)
            pred = self.fc(lstm_flat)  # Map down to single future vector element
            outputs.append(pred)
            
        return torch.cat(outputs, dim=1)  # Consolidated shape output: [Batch, Nodes]

In [7]:
tensors=torch.load('/kaggle/input/datasets/anubhashukla/prediction-phase/gnn_tensors (1).pt')
x_test,y_test=tensors['x_test'],tensors['y_test']
adj=np.load('/kaggle/input/datasets/anubhashukla/prediction-phase/spatial_weight_matrix (1).npy')

In [9]:
model=SpatioTemporalGNN(nodes=30,input_features=1,lookback=12,hidden_dims=16)
model.load_state_dict(torch.load('/kaggle/input/datasets/anubhashukla/prediction-phase/stgnn_model.pth'))
model.eval()

with torch.no_grad():
    gnn_preds=model(x_test,adj).numpy()

actuals=y_test.numpy()
var_preds=pd.read_csv('/kaggle/input/datasets/anubhashukla/stats-model-prediction/predictions.csv',index_col=0).values[-len(preds):]

def calculate_MASE(true_vals,pred_vals,training_history):
    mae_forecast=mean_absolute_error(true_vals,pred_vals)
    mae_naive = np.mean(np.abs(np.diff(training_history_slice)))
    if mae_naive == 0:
        return 1.0
    return mae_forecast / mae_naive

# Compute global aggregates across all 30 nodes over the entire test horizon
gnn_rmse = np.sqrt(mean_squared_error(actuals, gnn_preds))
var_rmse = np.sqrt(mean_squared_error(actuals, var_preds))

gnn_mae = mean_absolute_error(actuals, gnn_preds)
var_mae = mean_absolute_error(actuals, var_preds)

gnn_r2 = r2_score(actuals, gnn_preds)
var_r2 = r2_score(actuals, var_preds)

# Calculate MASE assuming a dummy persistence lag factor threshold baseline
gnn_mase = gnn_mae / (np.mean(np.abs(np.diff(actuals, axis=0))) + 1e-8)
var_mase = var_mae / (np.mean(np.abs(np.diff(actuals, axis=0))) + 1e-8)

print(f"{'Evaluation Metric':<25} | {'VAR Baseline':<12} | {'Spatio-Temporal GNN':<18}")
print(f"{'RMSE (Lower is Better)':<25} | {var_rmse:<12.5f} | {gnn_rmse:<18.5f}")
print(f"{'MAE (Lower is Better)':<25} | {var_mae:<12.5f} | {gnn_mae:<18.5f}")
print(f"{'MASE (Lower is Better)':<25} | {var_mase:<12.5f} | {gnn_mase:<18.5f}")
print(f"{'R-Squared (Higher is Better)':<25} | {var_r2:<12.5f} | {gnn_r2:<18.5f}")

#np.savez('model_evaluations.npz',actuals=actuals,gnn=preds,var=predictions)

Evaluation Metric         | VAR Baseline | Spatio-Temporal GNN
RMSE (Lower is Better)    | 0.23309      | 0.09486           
MAE (Lower is Better)     | 0.17271      | 0.06744           
MASE (Lower is Better)    | 4.05153      | 1.58213           
R-Squared (Higher is Better) | -0.33584     | 0.44083           
